# Build Indexes on Kaggle (GPU T4x2)Upload duy nhat file corpus.jsonl lam Kaggle Dataset.Notebook tu dong cai deps, load Harrier, build FAISS + BM25.

## 1. Setup

In [ ]:
import os, sys, json, pickle, re, math
from pathlib import Path

HF_CACHE = "/kaggle/working/hf_cache"
OUTPUT = "/kaggle/working/indexes"
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Tim corpus tu input dataset
INPUT = "/kaggle/input"
datasets = sorted(Path(INPUT).iterdir())
if not datasets:
    raise RuntimeError("No input dataset! Add corpus.jsonl via Add Input.")

corpus_dir = str(datasets[0])
corpus_files = list(Path(corpus_dir).rglob("*.jsonl"))
if not corpus_files:
    raise RuntimeError(f"No .jsonl found in {corpus_dir}")
CORPUS_PATH = str(corpus_files[0])
print(f'Corpus: {CORPUS_PATH}')

In [ ]:
line_count = int(os.popen(f'wc -l "{CORPUS_PATH}"').read().split()[0])
size_gb = os.path.getsize(CORPUS_PATH) / 1e9
print(f'Docs: {line_count:,}, Size: {size_gb:.1f} GB')

## 2. Install dependencies

In [ ]:
# Fix 1: torchvision 0.25.0 (Kaggle base) incompatible with torch 2.6.0
# Fix 2: pillow 12.2.0 (Kaggle base) incompatible with torchvision 0.21.0 → downgrade pillow
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124 --quiet --force-reinstall
!pip install "pillow<12" --quiet
!pip install faiss-gpu --quiet
!pip install transformers sentencepiece datasets duckdb pyarrow pandas tqdm
!pip install bm25s

In [ ]:
import torch
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Devices: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Load corpus

In [ ]:
from tqdm import tqdm

texts = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc='Loading', unit='docs'):
        doc = json.loads(line)
        texts.append(doc.get("text", ""))

print(f'Loaded {len(texts):,} texts')

## 4. Build Dense Index (FAISS + Harrier)

In [ ]:
from transformers import AutoModel, AutoTokenizer
import numpy as np

MODEL = "mainguyen9/vietlegal-harrier-0.6b"
print(f'Loading Harrier: {MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL, trust_remote_code=True, dtype=torch.float16).to('cuda')
model.eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

In [ ]:
import faiss
import torch.nn as nn

# Batch 32 → 33K iterations × ~1.5s = ~14h (trên Kaggle timeout 12h)
# Batch 256 → 4.1K iterations × ~4s = ~4.5h — T4 16GB vừa VRAM
# DataParallel: chia batch đều cho 2 GPU T4 → ~2.5h
BATCH_SIZE = 256
DIM = 1024

if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs via DataParallel')
    model = nn.DataParallel(model)

all_embs = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Encoding'):
    batch = texts[i:i+BATCH_SIZE]
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**inputs)
        # LAST-TOKEN POOLING (Qwen3 decoder) — phải khớp src/embedding/harrier_embedding.py
        # DataParallel có thể trả list[ModelOutput]; gộp về cuda:0 rồi gather token cuối non-pad
        if isinstance(outputs, (list, tuple)):
            last_hidden = torch.cat([o.last_hidden_state.to('cuda:0') for o in outputs], dim=0)
        else:
            last_hidden = outputs.last_hidden_state
        seq_len = inputs['attention_mask'].sum(dim=1) - 1
        emb = last_hidden[torch.arange(last_hidden.shape[0], device=last_hidden.device), seq_len]
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    all_embs.append(emb.cpu().numpy())
    if i % 5000 == 0 and i > 0:
        torch.cuda.empty_cache()

emb_array = np.vstack(all_embs).astype(np.float32)

index = faiss.IndexFlatIP(DIM)
index.add(emb_array)
print(f'FAISS index: {index.ntotal} vectors')

faiss.write_index(index, f'{OUTPUT}/dense.index')
index_size = os.path.getsize(f'{OUTPUT}/dense.index') / 1e9
print(f'Saved: {OUTPUT}/dense.index ({index_size:.1f} GB)')

# === FREE MEMORY cho BM25 === 
# texts ~2.5GB, emb_array ~4GB, model ~1.2GB, all_embs ~4GB → giải phóng trước khi BM25
del texts, all_embs, emb_array, model, tokenizer, index
import gc; gc.collect()
print('Memory freed. Ready for BM25.')

## 5. Build Sparse Index (BM25)

In [ ]:
import bm25s
import gc, unicodedata

# bm25s 0.3.9 không hỗ trợ tokenizer param → pre-process text rồi cho bm25s tự split
# Giống prepare_for_bm25(): NFKC + lowercase + abbreviation expansion
LEGAL_TERM_MAP = {
    'ld': 'lao động', 'lđ': 'lao động', 'blđ': 'bộ luật lao động',
    'bl': 'bộ luật', 'nd': 'nghị định', 'tt': 'thông tư',
    'cp': 'chính phủ', 'qđ': 'quyết định', 'ct': 'chỉ thị',
    'hđ': 'hợp đồng', 'tn': 'thu nhập', 'bh': 'bảo hiểm',
    'bhxh': 'bảo hiểm xã hội', 'bhyt': 'bảo hiểm y tế',
    'tncn': 'thu nhập cá nhân', 'gtgt': 'giá trị gia tăng',
    'tdn': 'thu nhập doanh nghiệp',
}
LEGAL_PAT = re.compile('|'.join(sorted(LEGAL_TERM_MAP, key=len, reverse=True)))

def preprocess(text: str) -> str:
    text = unicodedata.normalize('NFKC', text).lower()
    return LEGAL_PAT.sub(lambda m: LEGAL_TERM_MAP[m.group(0)], text)

# Step 1: Load + preprocess texts (NFKC + abbreviation expansion)
corpus_texts = []
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc='Loading', total=1_064_169):
        text = json.loads(line).get('text', '')
        corpus_texts.append(preprocess(text))

# Step 2: bm25s.tokenize() tự lower + split (idempotent vì đã lower rồi)
print(f'Tokenizing {len(corpus_texts):,} docs...')
tokenized = bm25s.tokenize(corpus_texts, show_progress=True)
del corpus_texts; gc.collect()

bm25 = bm25s.BM25()
bm25.index(tokenized)
del tokenized; gc.collect()

bm25.save(OUTPUT)
print(f'Saved BM25 index to {OUTPUT}')
print(f'  Vocab: {len(bm25.vocab):,} unique terms')

In [ ]:
print('=' * 50)
print('DONE!')
print('=' * 50)
for f in sorted(os.listdir(OUTPUT)):
    fp = f'{OUTPUT}/{f}'
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / 1e6
        print(f'  {f}: {size_mb:.0f} MB')